## Creating MCP- Langchain agent for accessing MongoDB 

In [1]:
import os 
from dotenv import load_dotenv 
load_dotenv() 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


import warnings

warnings.filterwarnings("ignore",category=DeprecationWarning)

from langchain.chat_models import init_chat_model

gemma = init_chat_model(model="gemma4:latest", model_provider="ollama")

#llm = init_chat_model(model="qwen/qwen3-32b", model_provider="Groq")
llm_primary = init_chat_model(model="llama-3.3-70b-versatile", model_provider="Groq")
llm_fallback_1 = init_chat_model(model="gpt-5.4-nano", model_provider="OpenAI")
llm_fallback_2 = init_chat_model(model="gpt-5.4-mini", model_provider="OpenAI")


In [2]:
MONGODB_URI = os.getenv("MONGODB_URI")

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

## Connect your client with the MongoDB-MCP-server 

In [4]:
import sys

In [5]:
client = MultiServerMCPClient({
# "mongodb":{
#     "transport":"stdio",
#     "command":"npx",
#     "args":[
#         "-y",
#         "mongodb-mcp-server@latest",
#         "--loggers",
#         "stderr"
#     ],
#     "env":{
#         "MDB_MCP_CONNECTION_STRING":MONGODB_URI
#     }
# },

"course_assistant": {
            "transport": "stdio",
            #"command": "/Users/ggmdnawazali/.local/bin/uv",
            "command":sys.executable,
            "args": [
                #"--directory",
                "/Users/nali/Documents/YTLLMs/MCP-Server/server.py",
                # "run",
                # "python",
                #"server.py",
            ],
        }

})

In [6]:
tools_mongoDB = await client.get_tools()

In [7]:
import sys

print(sys.executable)

/Users/nali/Documents/YTLLMs/.venv/bin/python


In [8]:
for tool in tools_mongoDB: 
    print(tool.name)

get_courses
get_course_details


## Create Langchain agent with mcp_tools

In [9]:
from langchain.agents import create_agent

prompt="""You are a helpful assistant. 
use tools_mongoDB for answering based on user query.
"""

agent= create_agent(
    model=  gemma,
    tools=tools_mongoDB,
    system_prompt=prompt
)

## Test the agent

In [10]:
user_query= """How many courses are there and what are the course codes?
 What are the software requirements for CS 466"""

In [11]:
from langchain.messages import SystemMessage,HumanMessage

try:
    result = await agent.ainvoke({
        "messages":[
            SystemMessage(content="You a helpful assistant."),
            HumanMessage(content=user_query)
        ]
    })
except Exception as e:
    print(f"Error happened during invoke. {e}")

In [12]:
print(result["messages"][-1].content)

Based on the available catalog information, there are at least **2** courses:

*   **CS 111:** AI for All - Artificial Intelligence for Life, Society, and Disciplines
*   **CS 466:** Natural Language Processing & Large Language Models (NLP & LLMs)

***

### Software Requirements for CS 466 (NLP & LLMs)

The course materials and tools required for CS 466 are:

*   zyBook for CS 466
*   VS Code
*   Python 3.x
*   PyTorch
*   Hugging Face Transformers and Datasets
*   LangChain and LangGraph
*   SentenceTransformers
*   FAISS or ChromaDB
*   **Computational Resources:** Local GPU, Google Colab, Kaggle, or university computing resources


In [13]:
user_query= """Show me the document information with field information:
name:James Cercone
for database: University, collection: students
"""

In [14]:
user_query= """Modify the following record:
current name:James Cercone modify to name: James Chase 
for database: University, collection: students
"""

In [15]:
user_query= """Add the following new document:
name:Ethan Nawaz 
dept_name:Computer Science
gpa:3.99
credit_hours:112 
for database: University, collection: students
"""

In [16]:
user_query= """Delete the following document:
name:Ethan Nawaz  
for database: University, collection: students
"""